# 💍 Jewellery Studio: Qwen-Image-Edit-2511 on free Kaggle (2× T4)

Takes a model photo and a jewellery product photo, and makes a photo of the model wearing that jewellery.
Comes with a page that works like the Hugging Face Space (Edit tab and Batch tab) and gives you a public `gradio.live` link.

**Kaggle settings:** Accelerator **GPU T4 x2** · Internet **On** · Secret **`HF_TOKEN`** (ticked) · dataset added through **Add Input**.
In the dataset, put model photos with `model` in the name (`0_model.png`, `1_model.png`), and use any name for the jewellery photos
(`necklace_01.jpg`, `earings_01.jpg`, ...).

**Order:** Cell 1, then **Run → Restart & clear cell outputs**, then Cells 2 → 5. After any restart, run Cells 2 → 5 again. Cell 1 isn't needed after a restart.

Tested on 2× T4: about **9.5 min per image** at 768×960 (8 steps). Presets: ⚡ Fast / ⚖️ Balanced / 💎 Best.

## 1. Install (once per session, then RESTART)

In [ ]:
# Versions that work together (checked against PyPI). diffusers 0.39 supports Qwen-Image-Edit-2511.
!pip install -q "diffusers==0.39.0" "transformers==4.57.1" "huggingface_hub>=0.34,<1.0" "gradio==5.50.0" accelerate peft bitsandbytes safetensors sentencepiece
!pip uninstall -y torchao        # Kaggle's old torchao blocks LoRA loading
import importlib.metadata as m
for p in ["diffusers", "transformers", "huggingface_hub", "gradio", "bitsandbytes", "peft", "accelerate"]:
    print(p, m.version(p))
print("\n👉 Now: Run → Restart & clear cell outputs, then run Cell 2")

## 2. HF token + bitsandbytes fix

In [ ]:
import os, importlib
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
print("✅ HF logged in as:", whoami()["name"])

# On Kaggle, diffusers sometimes fails to detect bitsandbytes. Tell it explicitly.
import bitsandbytes as bnb
import diffusers.utils.import_utils as d_iu
d_iu._bitsandbytes_available = True
d_iu._bitsandbytes_version = bnb.__version__
import transformers.utils.import_utils as t_iu
if hasattr(t_iu, "_bitsandbytes_available"):
    t_iu._bitsandbytes_available = True
for name in ["diffusers.quantizers.bitsandbytes.utils", "diffusers.quantizers.bitsandbytes.bnb_quantizer",
             "transformers.integrations.bitsandbytes", "transformers.quantizers.quantizer_bnb_4bit"]:
    try: importlib.import_module(name).bnb = bnb
    except Exception: pass
from diffusers.utils import is_bitsandbytes_available
print("✅ bitsandbytes visible to diffusers:", is_bitsandbytes_available())

## 3. Load the model on both GPUs (~3–4 min, ~33 GB download the first time)
* **GPU 0:** image model (4-bit), layers 0–35
* **GPU 1:** image model layers 36–59, text encoder (original, 4-bit, vision part kept at full precision) and VAE

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import glob, math, gc, time, random, shutil
import torch, torch.nn.functional as F
from PIL import Image, ImageOps
from accelerate.hooks import add_hook_to_module, AlignDevicesHook
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from diffusers import (QwenImageEditPlusPipeline, QwenImageTransformer2DModel,
                       AutoencoderKLQwenImage, FlowMatchEulerDiscreteScheduler)
import diffusers.pipelines.qwenimage.pipeline_qwenimage_edit_plus as qp

REPO_4BIT = "ovedrive/Qwen-Image-Edit-2511-4bit"   # image model (4-bit)
REPO_ORIG = "Qwen/Qwen-Image-Edit-2511"            # text encoder + VAE (original files)
OUTPUT_DIR = "/kaggle/working/outputs"; os.makedirs(OUTPUT_DIR, exist_ok=True)
GPU_IMG, GPU_TXT = torch.device("cuda:0"), torch.device("cuda:1")
N_MOVE = 24              # image-model layers moved to GPU 1 (20 worked; 24 leaves more room on GPU 0)
REF_PX = 768             # size the input photos are processed at (pipeline default 1024 = out of memory on T4)

assert torch.cuda.device_count() >= 2, "❌ Need GPU T4 x2 (Settings → Accelerator)"

def health(model, label):
    empty = [n for n, p in model.named_parameters() if p.device.type == "meta"]
    devs = sorted({str(p.device) for p in model.parameters()})
    print(f"   {label}: on {devs} | empty weights: {len(empty)}")
    assert not empty, f"❌ {label} has empty weights, e.g. {empty[:3]}"

def mem():
    return f"GPU0 {torch.cuda.memory_allocated(0)/1e9:.1f} GB | GPU1 {torch.cuda.memory_allocated(1)/1e9:.1f} GB"

if "_orig" not in globals() or getattr(_orig, "__name__", "") != "scaled_dot_product_attention":
    _orig = F.scaled_dot_product_attention if not getattr(F, "_patched", False) else None
assert _orig is not None and _orig.__name__ == "scaled_dot_product_attention", "restart the kernel and run Cell 3 again"

def _h(t): return t.half() if torch.is_tensor(t) and t.dtype == torch.bfloat16 else t

def _trivial_mask(m):
    if m is None: return True
    if m.dtype == torch.bool: return bool(m.all())
    return bool((m == 0).all())

def _sdpa(query, key, value, attn_mask=None, *args, **kwargs):
    if not (query.is_cuda and query.dtype == torch.bfloat16):
        return _orig(query, key, value, attn_mask, *args, **kwargs)
    q, k, v = _h(query), _h(key), _h(value)
    kw = {n: _h(x) for n, x in kwargs.items()}
    causal = kwargs.get("is_causal", False) or (len(args) >= 2 and bool(args[1]))
    if _trivial_mask(attn_mask):
        out = _orig(q, k, v, None, *args, **kw)
    elif causal or q.dim() != 4:
        out = _orig(q, k, v, _h(attn_mask), *args, **kw)
    else:
        m, L, outs = _h(attn_mask), q.shape[-2], []
        for i in range(0, L, 1024):
            mi = m[..., i:i + 1024, :] if (m.dim() >= 2 and m.shape[-2] == L) else m
            outs.append(_orig(q[..., i:i + 1024, :], k, v, mi, *args, **kw))
        out = torch.cat(outs, dim=-2)
    return out.to(torch.bfloat16)

F.scaled_dot_product_attention = _sdpa
torch.nn.functional.scaled_dot_product_attention = _sdpa
F._patched = True
print("✅ memory-safe fp16 attention active")

print("1/4 image model …")
torch.cuda.set_device(0)
transformer = QwenImageTransformer2DModel.from_pretrained(REPO_4BIT, subfolder="transformer", torch_dtype=torch.bfloat16)
health(transformer, "transformer"); print("  ", mem())

print("2/4 text encoder (4-bit, vision part full precision) …")
text_encoder = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    REPO_ORIG, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map={"": 1},
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["visual"]))
health(text_encoder, "text_encoder"); print("  ", mem())

print("3/4 VAE …")
vae = AutoencoderKLQwenImage.from_pretrained(REPO_ORIG, subfolder="vae", torch_dtype=torch.bfloat16).to(GPU_TXT)
health(vae, "vae")
pipe = QwenImageEditPlusPipeline.from_pretrained(
    REPO_4BIT, transformer=transformer, text_encoder=text_encoder, vae=vae, torch_dtype=torch.bfloat16)

print("4/4 Lightning 8-step LoRA …")
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config({
    "base_image_seq_len": 256, "base_shift": math.log(3), "invert_sigmas": False, "max_image_seq_len": 8192,
    "max_shift": math.log(3), "num_train_timesteps": 1000, "shift": 1.0, "shift_terminal": None,
    "stochastic_sampling": False, "time_shift_type": "exponential", "use_beta_sigmas": False,
    "use_dynamic_shifting": True, "use_exponential_sigmas": False, "use_karras_sigmas": False})
try:
    pipe.load_lora_weights("lightx2v/Qwen-Image-Edit-2511-Lightning",
                           weight_name="Qwen-Image-Edit-2511-Lightning-8steps-V1.0-bf16.safetensors")
    DEFAULT_STEPS, DEFAULT_CFG = 8, 1.0
    print("   ✅ Lightning loaded")
except Exception as e:
    print("   ⚠️ Lightning failed, normal mode (much slower):", repr(e))
    pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(REPO_ORIG, subfolder="scheduler")
    DEFAULT_STEPS, DEFAULT_CFG = 30, 4.0

# ---- device wiring ----
QwenImageEditPlusPipeline._execution_device = property(lambda self: GPU_TXT)   # pipeline runs on GPU 1
add_hook_to_module(pipe.transformer, AlignDevicesHook(execution_device=GPU_IMG, io_same_device=True))
blocks = pipe.transformer.transformer_blocks
for i in range(len(blocks) - N_MOVE, len(blocks)):                                # last layers -> GPU 1
    blocks[i].to(GPU_TXT)
    add_hook_to_module(blocks[i], AlignDevicesHook(execution_device=GPU_TXT, io_same_device=False))
for mod in [pipe.transformer.norm_out, pipe.transformer.proj_out]:                # output layers on GPU 0
    add_hook_to_module(mod, AlignDevicesHook(execution_device=GPU_IMG, io_same_device=False))
qp.VAE_IMAGE_SIZE = REF_PX * REF_PX
try: pipe.vae.enable_tiling()
except Exception: pass
pipe.set_progress_bar_config(disable=False)

gc.collect(); torch.cuda.empty_cache()
health(pipe.transformer, "transformer (split over 2 GPUs)")
print("\npipeline device:", pipe._execution_device, "|", mem())
print(f"✅ Model ready | steps={DEFAULT_STEPS}")

## 4. Dataset, prompts, helpers + test ONE image (~10 min)

In [ ]:
# ===================== Cell 4: prompts, auto-describe, auto-crop, quality helpers =====================
import glob, os, math, gc, time, random, shutil, re
import numpy as np
from PIL import Image, ImageOps
import diffusers.pipelines.qwenimage.pipeline_qwenimage_edit_plus as qp
RUN_TEST = True           # True = make one test image at the end of this cell

IMG_EXT = (".jpg", ".jpeg", ".png", ".webp")
DATA_IMGS  = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.lower().endswith(IMG_EXT))
MODEL_IMGS = [p for p in DATA_IMGS if "model" in os.path.basename(p).lower()]
JEWEL_IMGS = [p for p in DATA_IMGS if p not in MODEL_IMGS]
name = lambda p: os.path.splitext(os.path.basename(p))[0]
print("👩 models   :", [name(p) for p in MODEL_IMGS])
print("💍 jewellery:", len(JEWEL_IMGS), "images")

# ---------------- prompts ----------------
JEWEL_DESC = {"necklace": "a necklace", "collar": "a short collar-style necklace",
              "set": "a jewellery set: a necklace and a matching pair of earrings",
              "earrings": "a pair of earrings", "ring": "a ring", "bangles": "bangles"}
PLACE = {
    "necklace": "Put this exact necklace around her neck, with the pendant resting at the centre of her chest just below "
                "the collarbone. Do not add earrings or any other jewellery. ",
    "collar":   "Put this exact necklace snugly around the base of her neck like a collar, resting on her collarbones, "
                "with the central pendant hanging on her upper chest. Do not add earrings or any other jewellery. ",
    "set":      "Put the necklace around her neck with the pendant resting at the centre of her chest, and the matching "
                "earrings on her earlobes (partly hidden by her hair if her hair covers an ear). ",
    "earrings": "Put this exact pair of earrings on her earlobes (partly hidden by her hair if her hair covers an ear). "
                "Do not add a necklace or any other jewellery. ",
    "ring":     "Put this exact ring on the ring finger of her visible hand. ",
    "bangles":  "Put these exact bangles on her visible wrist. ",
}
REALISM = ("The jewellery must look really worn and photographed in this scene, not pasted on: it follows the curve of "
           "her neck and collarbones and the angle of her body, it rests naturally on her skin and goes slightly under "
           "the edge of her clothes where they overlap, it casts small soft contact shadows on her skin, and it is lit by "
           "the same light as Picture 1 (same direction, warmth and brightness) with natural metal reflections and gentle "
           "stone sparkle. Keep its real size, thickness and proportions relative to her neck as in Picture 2: delicate "
           "pieces stay delicate, heavy bridal pieces stay bold and substantial. Natural warm metal colour exactly as in "
           "Picture 2, not over-saturated. Same camera angle, focus, grain and colour tone as Picture 1. ")
KEEP = ("Keep her face, identity, expression, hairstyle, skin, clothes, pose and background exactly the same. "
        "Ignore the product background, props and any logo or watermark in Picture 2. ")

def make_prompt(jtype, desc=""):
    d = (desc or "").strip() or JEWEL_DESC[jtype]
    return (f"Picture 1 is a photo of a woman. Picture 2 is a product photo of {d}. " + PLACE[jtype] +
            "Copy the jewellery design from Picture 2 exactly, keeping every small detail: same shapes, granulation, "
            "beads, engraving, flowers, stones and their colours, enamel, drops and fringes. " + REALISM + KEEP +
            "Photorealistic professional jewellery photoshoot, high detail, sharp focus on the jewellery.")
DEFAULT_PROMPT = make_prompt("necklace")

# 2nd pass: compare the result with the product photo again and fix details + realism
DETAIL_PROMPT = ("Picture 1 is a photo of a woman wearing jewellery. Picture 2 is the original product photo of that "
                 "jewellery. Make the jewellery she is wearing match Picture 2 exactly in every small detail: the same "
                 "shapes, granulation, beads, engraving, flowers, stones and their colours, drops and fringes, and the same "
                 "natural metal colour. Make it look really worn in this scene: soft contact shadows on her skin, "
                 "reflections and lighting that match the room, following the curve of her neck. Do not change anything "
                 "else: her face, hair, clothes, pose and background stay exactly the same. High detail, photorealistic.")

def guess_type(path):
    f = os.path.basename(path).lower()
    if "set" in f:                                    return "set"
    if "choker" in f or "collar" in f or "haar" in f: return "collar"
    if "ear" in f:                                    return "earrings"      # earring_ / earings_
    if "ring" in f:                                   return "ring"
    if "bangle" in f or "bracelet" in f:              return "bangles"
    return "necklace"

# ---------------- vision helpers (use the already-loaded Qwen2.5-VL on GPU 1) ----------------
def _vlm(img, question, max_new_tokens=180):
    proc = pipe.processor
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    text = proc.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = proc(text=[text], images=[img], return_tensors="pt").to(GPU_TXT)
    with torch.inference_mode():
        ids = pipe.text_encoder.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.05)
    out = proc.batch_decode(ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    gc.collect(); torch.cuda.empty_cache()
    return " ".join(out.strip().split())

_ITEM = {"necklace": "necklace (ignore any earrings in the photo)", "collar": "necklace (ignore any earrings in the photo)",
         "set": "necklace and its matching earrings", "earrings": "pair of earrings", "ring": "ring", "bangles": "bangles"}
DESCRIBE_Q = ("You are a jewellery catalogue expert. Look very carefully at the {item} in this product photo and describe "
              "ONLY it in one detailed sentence that starts with 'a' (no intro, no list, no brand names). Mention: metal "
              "and colour (e.g. 22k yellow gold, rose gold, silver), style (e.g. traditional Indian, temple, Kolkata, "
              "modern, minimal), overall shape and how it is worn (choker, collar, short, long chain, mangalsutra), width "
              "and weight (delicate, medium, heavy, bold), the main pendant or motif (shape, size, design), stones (type, "
              "colour, where), beads, enamel, granulation, engraving, fringes or hanging drops, and the finish (high "
              "polish, matte, textured). Ignore the background, props, packaging and any logo.")

def describe_jewellery(img, jtype="necklace"):
    small = img.copy(); small.thumbnail((640, 640))
    out = _vlm(small, DESCRIBE_Q.format(item=_ITEM.get(jtype, "jewellery"))).rstrip(".")
    return out[0].lower() + out[1:] if out else ""

_CROP_ITEM = {"set": "necklace together with its earrings", "earrings": "pair of earrings", "ring": "ring",
              "bangles": "bangles"}
def auto_crop(img, jtype="necklace", margin=0.06):
    """Ask Qwen2.5-VL where the jewellery is and crop to it (removes props, logo, empty background).
    The model then sees the jewellery at a much higher resolution, so fine details survive."""
    small = img.copy(); small.thumbnail((672, 672))
    W, H = max(28, small.width // 28 * 28), max(28, small.height // 28 * 28)
    small = small.resize((W, H))
    item = _CROP_ITEM.get(jtype, "necklace (not the earrings)")
    try:
        ans = _vlm(small, f"Locate the {item} in this image. Answer only with JSON like "
                          '{"bbox_2d": [x1, y1, x2, y2]} in pixel coordinates.', 60)
        x1, y1, x2, y2 = [float(v) for v in re.findall(r"-?\d+(?:\.\d+)?", ans)[:4]]
    except Exception:
        return img, False
    if x2 <= x1 or y2 <= y1: return img, False
    sx, sy = img.width / W, img.height / H
    x1, x2, y1, y2 = x1 * sx, x2 * sx, y1 * sy, y2 * sy
    mx, my = (x2 - x1) * margin, (y2 - y1) * margin
    box = (max(0, int(x1 - mx)), max(0, int(y1 - my)), min(img.width, int(x2 + mx)), min(img.height, int(y2 + my)))
    if (box[2] - box[0]) * (box[3] - box[1]) < 0.05 * img.width * img.height: return img, False   # silly answer
    return img.crop(box), True

# ---------------- image helpers ----------------
def load_rgb(p): return ImageOps.exif_transpose(Image.open(p)).convert("RGB")

def trim_borders(img, thr=20, frac=0.05):
    """Cut black bars (e.g. a portrait photo padded to a square)."""
    g = np.asarray(img.convert("L")) > thr
    cols = np.where(g.mean(0) > frac)[0]; rows = np.where(g.mean(1) > frac)[0]
    if len(cols) == 0 or len(rows) == 0: return img
    box = (int(cols[0]), int(rows[0]), int(cols[-1]) + 1, int(rows[-1]) + 1)
    if (box[2] - box[0]) < 0.97 * img.width or (box[3] - box[1]) < 0.97 * img.height:
        return img.crop(box)
    return img

def size_for(img, area):
    w, h = img.size; s = math.sqrt(area / (w * h))
    return round(w*s/16)*16, round(h*s/16)*16

# Different processing size per input photo: model photo at REF, jewellery photo at a HIGHER size (keeps fine detail)
_SENT = 123457
if not hasattr(qp, "_orig_calculate_dimensions"):
    qp._orig_calculate_dimensions = qp.calculate_dimensions
_VAE_AREAS = []
def _calc_dims(target_area, ratio):
    if target_area == _SENT:
        target_area = _VAE_AREAS.pop(0) if _VAE_AREAS else 768 * 768
    return qp._orig_calculate_dimensions(target_area, ratio)
qp.calculate_dimensions = _calc_dims
qp.VAE_IMAGE_SIZE = _SENT

def run(images, prompt, seed, steps=None, area=768*960, ref_px=768, jewel_px=None, n=1):
    images = [trim_borders(images[0])] + list(images[1:])
    _VAE_AREAS[:] = [int(ref_px) ** 2] + [int(jewel_px or ref_px) ** 2] * (len(images) - 1)
    W, H = size_for(images[0], area)
    try:
        with torch.inference_mode():
            out = pipe(image=images, prompt=prompt, negative_prompt=" ", width=W, height=H,
                       num_inference_steps=int(steps or DEFAULT_STEPS), true_cfg_scale=DEFAULT_CFG,
                       num_images_per_prompt=int(n), generator=torch.Generator("cpu").manual_seed(int(seed))).images
    finally:
        _VAE_AREAS.clear(); gc.collect(); torch.cuda.empty_cache()
    return [im.convert("RGB") for im in out]

def safe_run(images, prompt, seed, steps, area, ref_px, jewel_px):
    """run(); if the GPU runs out of memory, redo it once at Balanced size instead of failing."""
    try:
        return run(images, prompt, seed, steps, area, ref_px, jewel_px)[0], ""
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        return (run(images, prompt, seed, steps, min(area, 768*960), min(int(ref_px), 768), 768)[0],
                " · ⚠️ out of memory, redone at a smaller size")

def refine(img, jewel, seed, steps=None, area=768*960, ref_px=768, jewel_px=None):
    """2nd pass: result + product photo -> fix the details and blend into the scene."""
    return run([img, jewel], DETAIL_PROMPT, seed, steps, area, ref_px, jewel_px)[0]

def side_by_side(*imgs, h=512):
    imgs = [i.resize((int(i.width*h/i.height), h)) for i in imgs]
    c = Image.new("RGB", (sum(i.width for i in imgs), h), "white"); x = 0
    for i in imgs: c.paste(i, (x, 0)); x += i.width
    return c

print("✅ helpers ready | jewellery types:", list(PLACE))
if RUN_TEST:
    m, jp = load_rgb(MODEL_IMGS[0]), JEWEL_IMGS[0]
    j, cropped = auto_crop(load_rgb(jp), guess_type(jp))
    print("auto-crop:", cropped, j.size)
    t = time.time(); out = run([m, j], make_prompt(guess_type(jp)), 0, jewel_px=1024)[0]
    print(f"⏱ {time.time()-t:.0f} s"); out.save(f"{OUTPUT_DIR}/test.png")
    display(side_by_side(j, trim_borders(m), out))

## 5. The Jewellery Studio page (public link)
Open the `https://xxxx.gradio.live` link it prints. Keep the Kaggle tab open while you use it.
**Download your ZIP before you stop the session**, because `/kaggle/working` is cleared when an interactive session ends.

In [ ]:
# ===================== Cell 5: Jewellery Studio page =====================
import gradio as gr
# close any page started earlier in this session (prevents "bound to a different event loop")
try: demo.close()
except Exception: pass
gr.close_all()
try:
    import gradio.routes as _gro, gradio.route_utils as _gru
    _gro.file_upload_statuses = _gru.FileUploadProgress()
except Exception as e:
    print("note: upload tracker not reset:", repr(e))

MAX_SEED = 2**31 - 1
TYPES = list(PLACE)
OUT_SIZES = {"Small 640×800": 640*800, "Medium 768×960": 768*960, "Large 896×1120": 896*1120, "XL 1024×1280": 1024*1280}
PX = {"512": 512, "640": 640, "768": 768, "1024": 1024}
#            name                      steps  output            model-photo  jewellery-photo  detail pass
PRESETS = {"⚡ Fast (~5 min)":        (8,    "Small 640×800",  "640",       "768",           False),
           "⚖️ Balanced (~10 min)":   (8,    "Medium 768×960", "768",       "1024",          False),
           "💎 Best (~25 min)":       (8,    "Large 896×1120", "768",       "1024",          True)}

def to_pils(items):
    out = []
    for it in items or []:
        x = it[0] if isinstance(it, (list, tuple)) else it
        if isinstance(x, dict): x = x.get("path") or x.get("name")
        out.append(x.convert("RGB") if isinstance(x, Image.Image) else load_rgb(x))
    return out

def fmt(sec): return f"{int(sec//60)} min {int(sec%60)} s"
def save_png(im, stem):
    p = f"{OUTPUT_DIR}/{stem}.png"; im.convert("RGB").save(p); return p
def apply_preset(p): return PRESETS[p]

def ui_describe(images, jtype):
    imgs = to_pils(images)
    if len(imgs) < 2: raise gr.Error("Add the model photo first and the jewellery photo second")
    j, _ = auto_crop(imgs[1], jtype)
    return describe_jewellery(j, jtype)

# ---------------- Tab 1: like the HF Space ----------------
def infer(images, prompt, jtype, do_crop, seed, randomize, steps, out_size, model_px, jewel_px, detail_pass,
          progress=gr.Progress(track_tqdm=True)):
    imgs = to_pils(images)
    if len(imgs) < 2: raise gr.Error("Add 2 images: 1st = model photo, 2nd = jewellery photo")
    if randomize: seed = random.randint(0, MAX_SEED)
    m, j = imgs[0], imgs[1]
    t, ts, results = time.time(), int(time.time()), []
    if do_crop:
        j, ok = auto_crop(j, jtype)
        results.append((save_png(j, f"edit_{ts}_crop"), "jewellery crop used" if ok else "jewellery (crop not found, full photo used)"))
        yield results, seed, "✂️ jewellery cropped · ⏳ generating…"
    a, mp, jp_ = OUT_SIZES[out_size], PX[model_px], PX[jewel_px]
    try:
        out, warn = safe_run([m, j], prompt, seed, steps, a, mp, jp_)
        results.append((save_png(out, f"edit_{ts}"), "result"))
        yield results, seed, f"✅ result ready in {fmt(time.time()-t)} · seed {seed}{warn}" + (" · ⏳ detail pass running…" if detail_pass else "")
        if detail_pass:
            out2, _ = safe_run([out, j], DETAIL_PROMPT, seed, steps, a, mp, jp_)
            results.append((save_png(out2, f"edit_{ts}_detail"), "after detail pass"))
            yield results, seed, f"✅ all done in {fmt(time.time()-t)} · seed {seed}"
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise gr.Error("Out of GPU memory. Pick a faster preset or a smaller size")

def latest(k=6):
    return sorted(glob.glob(f"{OUTPUT_DIR}/edit_*.png"), key=os.path.getmtime)[-k:][::-1]

# ---------------- Tab 2: batch ----------------
def batch(model_choice, model_upload, jewel_names, extra_files, auto_type, jtype, desc, auto_desc, do_crop,
          prompt_override, extra_text, seed, randomize, steps, out_size, model_px, jewel_px, detail_pass, skip,
          progress=gr.Progress(track_tqdm=True)):
    if model_upload is not None:
        models = [("uploaded", model_upload.convert("RGB"))]
    elif model_choice == "All models":
        models = [(name(p), load_rgb(p)) for p in MODEL_IMGS]
    else:
        models = [(model_choice, load_rgb(next(p for p in MODEL_IMGS if name(p) == model_choice)))]
    jewels = [p for p in JEWEL_IMGS if name(p) in (jewel_names or [])]
    jewels += [f if isinstance(f, str) else f.name for f in (extra_files or [])]
    if not models or not jewels: raise gr.Error("Pick at least one model and one jewellery image")

    outdir = f"{OUTPUT_DIR}/batch"; os.makedirs(outdir, exist_ok=True)
    a, mp, jp_ = OUT_SIZES[out_size], PX[model_px], PX[jewel_px]
    jobs = [(mn, mi, jp) for mn, mi in models for jp in jewels]
    prepared = {}                                   # crop + description once per jewellery photo
    gallery, done_t, t0 = [], [], time.time()
    for k, (mn, mi, jp) in enumerate(jobs, 1):
        out = f"{outdir}/{name(jp)}__{mn}.png"
        if not (skip and os.path.exists(out)):
            jt = guess_type(jp) if auto_type else jtype
            if jp not in prepared:
                j = load_rgb(jp)
                if do_crop: j, _ = auto_crop(j, jt)
                d = desc.strip() or (describe_jewellery(j, jt) if auto_desc else "")
                prepared[jp] = (j, d)
            j, d = prepared[jp]
            p = prompt_override.strip() or make_prompt(jt, d)
            if extra_text.strip(): p += " " + extra_text.strip()
            s = random.randint(0, MAX_SEED) if randomize else int(seed)
            t = time.time()
            try:
                im, _ = safe_run([mi, j], p, s, steps, a, mp, jp_)
                if detail_pass: im, _ = safe_run([im, j], DETAIL_PROMPT, s, steps, a, mp, jp_)
                im.save(out); open(out[:-4] + ".txt", "w").write(p); done_t.append(time.time() - t)
            except Exception as e:
                print("failed", out, repr(e)); gc.collect(); torch.cuda.empty_cache(); continue
        gallery = (gallery + [(out, f"{name(jp)} · {mn}")])[-24:]
        eta = (sum(done_t)/len(done_t)) * (len(jobs) - k) if done_t else 0
        yield gallery, None, f"⏳ {k}/{len(jobs)} done · elapsed {fmt(time.time()-t0)} · about {fmt(eta)} left"
    zip_path = shutil.make_archive(f"{OUTPUT_DIR}/jewellery_results", "zip", outdir)
    yield gallery, zip_path, f"✅ Finished {len(jobs)} images in {fmt(time.time()-t0)}. Download the ZIP below"

def quality_block(default="⚖️ Balanced (~10 min)"):
    preset = gr.Radio(list(PRESETS), value=default, label="Quality preset")
    with gr.Accordion("Advanced settings", open=False):
        s, o, mp, jp, dp = PRESETS[default]
        steps = gr.Slider(4, 12, value=s, step=1, label="Steps")
        with gr.Row():
            out_size = gr.Dropdown(list(OUT_SIZES), value=o, label="Output size")
            model_px = gr.Dropdown(list(PX), value=mp, label="Model-photo processing size")
            jewel_px = gr.Dropdown(list(PX), value=jp, label="Jewellery-photo processing size (higher = more detail)")
        detail = gr.Checkbox(value=dp, label="💎 Detail pass: 2nd run that compares with the product photo and fixes details + lighting (doubles time)")
        with gr.Row():
            seed = gr.Slider(0, MAX_SEED, value=0, step=1, label="Seed")
            rnd = gr.Checkbox(value=True, label="Randomize seed")
    preset.change(apply_preset, preset, [steps, out_size, model_px, jewel_px, detail])
    return steps, out_size, model_px, jewel_px, detail, seed, rnd

with gr.Blocks(title="Jewellery Studio") as demo:
    gr.Markdown("# 💍 Jewellery Studio · Qwen-Image-Edit-2511\n"
                "**For the best results:** model photo with the neck clearly visible · jewellery photo of ONE product "
                "(auto-crop removes the background) · click 👁 Auto-describe and check the text · use 💎 Best for final images.")

    with gr.Tab("Edit (like HF Space)"):
        with gr.Row():
            inp = gr.Gallery(label="Input: 1st = model photo, 2nd = jewellery photo", type="pil", interactive=True, height=450)
            res = gr.Gallery(label="Result", type="filepath", format="png", height=450)
        with gr.Row():
            jtype = gr.Dropdown(TYPES, value="necklace", label="Jewellery type")
            desc = gr.Textbox(label="Describe the jewellery (click 👁 Auto-describe, then edit if needed)", scale=3,
                              placeholder="e.g. a heavy 22k yellow-gold collar necklace with granulated beads and a round ruby pendant")
            with gr.Column(scale=1):
                auto = gr.Button("👁 Auto-describe")
                build = gr.Button("↻ Build prompt")
        prompt = gr.Textbox(value=DEFAULT_PROMPT, label="Prompt (you can edit it)", lines=5)
        build.click(make_prompt, [jtype, desc], prompt)
        auto.click(ui_describe, [inp, jtype], desc).then(make_prompt, [jtype, desc], prompt)
        do_crop = gr.Checkbox(value=True, label="✂️ Auto-crop the jewellery photo (recommended)")
        steps, out_size, model_px, jewel_px, detail, seed, rnd = quality_block()
        btn = gr.Button("✨ Edit!", variant="primary")
        info = gr.Markdown()
        btn.click(infer, [inp, prompt, jtype, do_crop, seed, rnd, steps, out_size, model_px, jewel_px, detail], [res, seed, info])
        gr.Button("🔄 Show latest results", size="sm").click(latest, None, res)

    with gr.Tab("Batch (model × jewellery)"):
        gr.Markdown(f"Dataset: **{len(MODEL_IMGS)} models**, **{len(JEWEL_IMGS)} jewellery**. "
                    "Saved to `/kaggle/working/outputs/batch` with the prompt as .txt (already-done images are skipped).")
        with gr.Row():
            bmodel_choice = gr.Dropdown(["All models"] + [name(p) for p in MODEL_IMGS],
                                        value=name(MODEL_IMGS[0]) if MODEL_IMGS else "All models", label="Model photo")
            bmodel_up = gr.Image(type="pil", label="…or upload another model photo", height=220)
        bjewels = gr.CheckboxGroup([name(p) for p in JEWEL_IMGS], value=[name(p) for p in JEWEL_IMGS],
                                   label="Jewellery to generate (untick to skip)")
        bextra = gr.File(file_count="multiple", file_types=["image"], type="filepath", label="Extra jewellery photos (optional)")
        with gr.Row():
            bauto = gr.Checkbox(value=True, label="Type from file name (set_, collar_, earings_, necklace_, ring_, bangle_)")
            btype = gr.Dropdown(TYPES, value="necklace", label="Type (when the box on the left is off)")
        with gr.Row():
            bcrop = gr.Checkbox(value=True, label="✂️ Auto-crop each jewellery photo")
            bautod = gr.Checkbox(value=True, label="👁 Auto-describe each jewellery photo")
        bdesc = gr.Textbox(label="Description for all items (optional, overrides auto-describe)")
        bprompt = gr.Textbox(label="Full prompt override (optional)", lines=2)
        bextra_text = gr.Textbox(label="Extra text added to every prompt (optional)")
        bsteps, bout, bmp, bjp, bdetail, bseed, brnd = quality_block()
        bskip = gr.Checkbox(value=True, label="Skip images already generated (resume)")
        bbtn = gr.Button("✨ Generate all", variant="primary")
        bstatus = gr.Textbox(label="Status")
        bgal = gr.Gallery(label="Latest results", columns=4, format="png", height=500)
        bzip = gr.File(label="Download all (ZIP)")
        bbtn.click(batch, [bmodel_choice, bmodel_up, bjewels, bextra, bauto, btype, bdesc, bautod, bcrop, bprompt,
                           bextra_text, bseed, brnd, bsteps, bout, bmp, bjp, bdetail, bskip], [bgal, bzip, bstatus])

demo.queue(default_concurrency_limit=1).launch(share=True, debug=True)

## Troubleshooting
| Problem | Fix |
|---|---|
| `CUDA out of memory` | Choose a smaller **Output size** or **Input-photo size** on the page |
| `FLAX_WEIGHTS_NAME` / import errors | You didn't restart after Cell 1. Restart, then run Cells 2 → 5 |
| `❌ … empty weights` in Cell 3 | Restart and run Cell 3 again. If it repeats, report the output |
| Jewellery design not exact | Use input-photo size 768, 8–10 steps, a tightly cropped product photo, and the "Extra text" box |
| Link stopped working | The session ended or went idle. Run Cells 2 → 5 again (no downloads needed) |